# Geração de Dados Fictícios - PNAB Ciclo 2 (Fase 2)

## Objetivo
Este notebook gera dados fictícios para a **Fase 2 (Relatório de Execução Cultural)** do padrão SNIIC - PNAB Ciclo 2.

Ele cria 2 DataFrames distintos, alinhados ao dicionário oficial da Fase 2:

1. `Ação Cultural Realizada`
2. `Entregas (Realizadas)`

Os dados produzidos servem para testes de importação, validação de layout, desenvolvimento de integrações e simulações de consolidação em planilhas ou APIs.

## O que o notebook faz
- carrega as listas de valores oficiais da PNAB a partir do arquivo local do projeto;
- usa `Faker` com localidade `pt_BR` para produzir dados realistas;
- respeita os nomes de campos descritos em `dicionario-fase-2.md`;
- gera relações plausíveis entre ações executadas e entregas realizadas;
- exporta os resultados em CSV com codificação `utf-8-sig`.

## Arquivos gerados
Ao executar o notebook, serão criados os arquivos:
- `fase2_acao_cultural_realizada.csv`
- `fase2_entregas_realizadas.csv`


In [ ]:
import json
import random
from datetime import date, datetime, timedelta
from pathlib import Path

import pandas as pd

try:
    from faker import Faker
except ImportError:
    %pip install faker
    from faker import Faker


In [5]:
# Configuração base para reprodutibilidade
SEED = 42
random.seed(SEED)
fake = Faker('pt_BR')
Faker.seed(SEED)

# Quantidade padrão de registros
NUM_ACOES = 50
NUM_ENTREGAS = 80


In [ ]:
# Carrega as listas de valores a partir do arquivo local do projeto
base_dir = Path.cwd()
json_path = base_dir / 'lista_de_valores.json'

if not json_path.exists():
    json_path = base_dir / 'arquivos_e_modelos' / 'lista_de_valores.json'

if not json_path.exists():
    raise FileNotFoundError(
        'Não foi possível localizar o arquivo lista_de_valores.json. '
        'Execute este notebook na pasta arquivos_e_modelos ou na raiz do projeto.'
    )

with open(json_path, 'r', encoding='utf-8') as f:
    listas_json = json.load(f)

print(f'Arquivo carregado com sucesso: {json_path}')
print('Listas disponíveis:')
print('\n'.join(listas_json.keys()))


In [7]:
# Listas complementares previstas no dicionário, mas não centralizadas no JSON
FORMATO_EXECUCAO = [
    'Presencialmente em local fixo',
    'Presencialmente itinerante',
    'Remotamente/Online',
    'Em formato híbrido',
    'Outros',
    'Não aplicável'
]

TIPOS_RESPONSAVEL = [
    'Pessoa física',
    'MEI',
    'Pessoa jurídica',
    'Coletivo informal'
]

MEDIDAS_ACESSIBILIDADE = (
    listas_json['15.1 Acessibilidade arquitetônica']
    + listas_json['15.2 Acessibilidade comunicacional']
    + listas_json['15.3 Acessibilidade atitudinal']
)


In [8]:
def formatar_data(dt: date) -> str:
    return dt.strftime('%d/%m/%Y')


def numero_monetario(minimo: float, maximo: float) -> float:
    return round(random.uniform(minimo, maximo), 2)


def gerar_periodo_execucao():
    hoje = date.today()
    data_recebimento = fake.date_between(
        start_date=hoje - timedelta(days=540),
        end_date=hoje - timedelta(days=180)
    )
    inicio = data_recebimento + timedelta(days=random.randint(7, 60))
    termino = inicio + timedelta(days=random.randint(1, 180))
    return data_recebimento, inicio, termino


def gerar_fontes_adicionais():
    fontes = listas_json['14. Recursos Financeiros de Outras Fontes']
    sem_outras_fontes = 'Não, a ação não possui outras fontes de recursos financeiros'

    if random.random() < 0.35:
        return sem_outras_fontes, 0.0

    opcoes = [item for item in fontes if item != sem_outras_fontes]
    selecionadas = random.sample(opcoes, k=random.randint(1, min(3, len(opcoes))))
    return ', '.join(selecionadas), numero_monetario(1000, 80000)


def gerar_cpfs(qtd: int) -> str:
    if qtd <= 0:
        return None

    cpfs = []
    vistos = set()
    while len(cpfs) < qtd:
        cpf = fake.cpf()
        if cpf not in vistos:
            vistos.add(cpf)
            cpfs.append(cpf)
    return ', '.join(cpfs)


def escolher_cep_por_formato(formato: str):
    if formato in {'Remotamente/Online', 'Não aplicável'}:
        return None
    if formato == 'Outros' and random.random() < 0.4:
        return None
    return fake.postcode()


def gerar_medidas_acessibilidade():
    if random.random() < 0.2:
        return None
    qtd = random.randint(1, 4)
    return ', '.join(random.sample(MEDIDAS_ACESSIBILIDADE, k=qtd))


def escolher_publico_por_tipo_entrega(tipo_entrega: str) -> int:
    tipos_intimistas = {
        'Oficina', 'Curso / Oficina / Formação', 'Palestra / Seminário / Debate',
        'Publicação impressa', 'Livro', 'Catálogo'
    }
    tipos_digitais = {
        'Aplicativo / Software', 'Site', 'Podcast', 'Vídeo', 'Documentário', 'Curta-metragem'
    }

    if tipo_entrega in tipos_intimistas:
        return random.randint(10, 200)
    if tipo_entrega in tipos_digitais:
        return random.randint(100, 5000)
    return random.randint(30, 1500)


In [9]:
def generate_acao_cultural_realizada(num_rows: int) -> pd.DataFrame:
    registros = []

    editais = [f'Edital-PNAB-2025-{i:03d}' for i in range(1, max(10, num_rows // 3) + 1)]

    for _ in range(num_rows):
        identificador_edital = random.choice(editais)
        tipo_responsavel = random.choice(TIPOS_RESPONSAVEL)
        data_recebimento, data_inicio, data_termino = gerar_periodo_execucao()
        valor_recebido = numero_monetario(3000, 150000)
        outras_fontes, valor_outras_fontes = gerar_fontes_adicionais()
        qtd_pessoas = random.randint(0, 20)
        formato = random.choice(FORMATO_EXECUCAO)

        registro = {
            'identificador_edital': identificador_edital,
            'cpf_responsavel_execucao': fake.cpf() if tipo_responsavel in {'Pessoa física', 'Coletivo informal'} else None,
            'cnpj_responsavel_execucao': fake.cnpj() if tipo_responsavel in {'MEI', 'Pessoa jurídica'} else None,
            'data_inicio_execucao': formatar_data(data_inicio),
            'data_termino_execucao': formatar_data(data_termino),
            'valor_recebido_pnab': valor_recebido,
            'data_recebimento_recurso': formatar_data(data_recebimento),
            'recebeu_recursos_outras_fontes': outras_fontes,
            'valor_recursos_outras_fontes': valor_outras_fontes,
            'qtd_pessoas_remuneradas_exec': qtd_pessoas,
            'cpfs_remunerados': gerar_cpfs(qtd_pessoas),
            'formato_execucao': formato,
            'cep_local_execucao': escolher_cep_por_formato(formato)
        }
        registros.append(registro)

    return pd.DataFrame(registros)



def generate_entregas_realizadas(num_rows: int, df_acoes: pd.DataFrame) -> pd.DataFrame:
    registros = []
    tipos_entrega = listas_json['12. Tipos de Entregas']
    segmentos = listas_json['8. Segmento Cultural']

    for _ in range(num_rows):
        acao = df_acoes.sample(1, random_state=random.randint(1, 1_000_000)).iloc[0]
        formato_execucao = acao['formato_execucao']
        tipo_entrega = random.choice(tipos_entrega)

        if formato_execucao in {'Remotamente/Online', 'Não aplicável'}:
            cep_entrega = None
        elif formato_execucao == 'Em formato híbrido':
            cep_entrega = random.choice([fake.postcode(), None])
        else:
            cep_entrega = fake.postcode()

        registro = {
            'identificador_edital_entrega': acao['identificador_edital'],
            'tipo_entrega': tipo_entrega,
            'segmento_cultural_entrega': random.choice(segmentos),
            'medidas_acessibilidade': gerar_medidas_acessibilidade(),
            'cep_local_entrega': cep_entrega,
            'publico_alcancado_entrega': escolher_publico_por_tipo_entrega(tipo_entrega)
        }
        registros.append(registro)

    return pd.DataFrame(registros)


In [ ]:
# Geração dos DataFrames
df_acao_cultural_realizada = generate_acao_cultural_realizada(NUM_ACOES)
df_entregas_realizadas = generate_entregas_realizadas(NUM_ENTREGAS, df_acao_cultural_realizada)

# Exportação dos CSVs
output_dir = Path.cwd()
df_acao_cultural_realizada.to_csv(output_dir / 'fase2_acao_cultural_realizada.csv', index=False, encoding='utf-8-sig')
df_entregas_realizadas.to_csv(output_dir / 'fase2_entregas_realizadas.csv', index=False, encoding='utf-8-sig')

print('Arquivos gerados com sucesso:')
print('-', output_dir / 'fase2_acao_cultural_realizada.csv')
print('-', output_dir / 'fase2_entregas_realizadas.csv')
print()
print('Dimensões:')
print('Ações culturais realizadas:', df_acao_cultural_realizada.shape)
print('Entregas realizadas:', df_entregas_realizadas.shape)


In [11]:
df_acao_cultural_realizada.head()

,identificador_edital,cpf_responsavel_execucao,cnpj_responsavel_execucao,data_inicio_execucao,data_termino_execucao,valor_recebido_pnab,data_recebimento_recurso,recebeu_recursos_outras_fontes,valor_recursos_outras_fontes,qtd_pessoas_remuneradas_exec,cpfs_remunerados,formato_execucao,cep_local_execucao
0,Edital-PNAB-2025-004,104.965.823-05,NaN,30/07/2025,09/10/2025,38999.10,06/06/2025,"Não, a ação não possui outras fontes de recurs...",0.00,3,"960.781.425-85, 836.194.205-05, 654.192.038-98",Não aplicável,NaN
1,Edital-PNAB-2025-003,594.603.817-66,NaN,16/11/2024,24/11/2024,16773.20,07/11/2024,"Não, a ação não possui outras fontes de recurs...",0.00,19,"845.910.623-33, 413.607.289-96, 258.179.043-14...",Presencialmente em local fixo,31165-667
2,Edital-PNAB-2025-007,019.352.764-25,NaN,22/09/2025,15/01/2026,89622.06,01/09/2025,Recursos de Lei de Incentivo Municipal,56153.01,10,"387.196.240-69, 179.054.632-06, 267.318.045-17...",Remotamente/Online,NaN
3,Edital-PNAB-2025-005,NaN,13.529.864/0001-47,02/11/2025,28/01/2026,18024.91,08/09/2025,"Patrocínio privado direto, Cobrança de ingressos",21897.15,1,804.591.263-42,Não aplicável,NaN
4,Edital-PNAB-2025-015,824.971.605-11,NaN,16/08/2025,06/09/2025,84149.97,16/07/2025,"Patrocínio privado direto, Cobrança de ingress...",56661.18,1,874.605.319-01,Não aplicável,NaN


In [12]:
df_entregas_realizadas.head()

,identificador_edital_entrega,tipo_entrega,segmento_cultural_entrega,medidas_acessibilidade,cep_local_entrega,publico_alcancado_entrega
0,Edital-PNAB-2025-008,Caderno / Cartilha / Apostila,Patrimônio Cultural Imaterial,"Iluminação adequada, Assentos para pessoas obe...",53942-373,1022
1,Edital-PNAB-2025-011,Desfile,Hip Hop,NaN,NaN,615
2,Edital-PNAB-2025-002,Livro,Dança,Corrimãos e guarda-corpos,11179080,48
3,Edital-PNAB-2025-011,Festa Popular,Patrimônio Cultural Material,NaN,59579-472,468
4,Edital-PNAB-2025-015,Filme de curta-metragem,Festas e Celebrações,"Audiodescrição, Piso tátil, Vagas de estaciona...",63920-867,1226
